# 15 — Model Artifact Validation

## Objective

Before continuing to FastAPI, services, guardrails and agents, verify that all three ML modules are complete:

1. Renewal
2. Fraud
3. Underwriting

This notebook checks:

- required `.joblib` files exist
- required curated model datasets exist
- model artifacts load correctly
- saved feature schemas exist
- one-row smoke predictions work
- thresholds / label encoder are valid

Run this notebook only after completing:

- `11_renewal_model.ipynb`
- `13_fraud_model.ipynb`
- `14_underwriting_model.ipynb`

In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

pd.set_option(
    "display.max_columns",
    None
)

print(
    "Validation libraries imported."
)

Validation libraries imported.


## 1. Resolve project paths

In [2]:
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists():
    PROJECT_DIR = CURRENT_DIR

elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_DIR = CURRENT_DIR.parent

else:
    raise FileNotFoundError(
        "Could not locate project root."
    )

MODELS_DIR = (
    PROJECT_DIR
    / "models"
)

CURATED_DIR = (
    PROJECT_DIR
    / "data"
    / "curated"
)

SRC_DIR = (
    PROJECT_DIR
    / "src"
)

print(
    "PROJECT_DIR:",
    PROJECT_DIR
)

PROJECT_DIR: f:\insurance_ai_copilot


## 2. Required artifact checklist

In [3]:
required_model_files = [
    "renewal_xgboost_model.joblib",
    "renewal_churn_threshold.joblib",

    "fraud_xgboost_model.joblib",
    "fraud_threshold.joblib",

    "underwriting_xgboost_model.joblib",
    "underwriting_label_encoder.joblib",
]

artifact_status = []

for file_name in required_model_files:

    path = (
        MODELS_DIR
        / file_name
    )

    artifact_status.append({
        "FILE":
            file_name,

        "EXISTS":
            path.exists(),

        "PATH":
            str(path),
    })


artifact_status_df = (
    pd.DataFrame(
        artifact_status
    )
)

display(
    artifact_status_df
)

assert (
    artifact_status_df[
        "EXISTS"
    ].all()
), (
    "Some required model artifacts are missing. "
    "Run fraud/underwriting training notebooks first."
)

,FILE,EXISTS,PATH
0,renewal_xgboost_model.joblib,True,f:\insurance_ai_copilot\models\renewal_xgboost...
1,renewal_churn_threshold.joblib,True,f:\insurance_ai_copilot\models\renewal_churn_t...
2,fraud_xgboost_model.joblib,True,f:\insurance_ai_copilot\models\fraud_xgboost_m...
3,fraud_threshold.joblib,True,f:\insurance_ai_copilot\models\fraud_threshold...
4,underwriting_xgboost_model.joblib,True,f:\insurance_ai_copilot\models\underwriting_xg...
5,underwriting_label_encoder.joblib,True,f:\insurance_ai_copilot\models\underwriting_la...


## 3. Required curated datasets

In [4]:
required_datasets = [
    "renewal_model_dataset.csv",
    "fraud_model_dataset.csv",
    "underwriting_model_dataset.csv",
]

dataset_status = []

for file_name in required_datasets:

    path = (
        CURATED_DIR
        / file_name
    )

    dataset_status.append({
        "FILE":
            file_name,

        "EXISTS":
            path.exists(),

        "PATH":
            str(path),
    })


dataset_status_df = (
    pd.DataFrame(
        dataset_status
    )
)

display(
    dataset_status_df
)

assert (
    dataset_status_df[
        "EXISTS"
    ].all()
), (
    "Some required curated datasets are missing."
)

,FILE,EXISTS,PATH
0,renewal_model_dataset.csv,True,f:\insurance_ai_copilot\data\curated\renewal_m...
1,fraud_model_dataset.csv,True,f:\insurance_ai_copilot\data\curated\fraud_mod...
2,underwriting_model_dataset.csv,True,f:\insurance_ai_copilot\data\curated\underwrit...


## 4. Load all six artifacts

In [5]:
renewal_model = joblib.load(
    MODELS_DIR
    / "renewal_xgboost_model.joblib"
)

renewal_threshold = joblib.load(
    MODELS_DIR
    / "renewal_churn_threshold.joblib"
)


fraud_model = joblib.load(
    MODELS_DIR
    / "fraud_xgboost_model.joblib"
)

fraud_threshold = joblib.load(
    MODELS_DIR
    / "fraud_threshold.joblib"
)


underwriting_model = joblib.load(
    MODELS_DIR
    / "underwriting_xgboost_model.joblib"
)

underwriting_label_encoder = joblib.load(
    MODELS_DIR
    / "underwriting_label_encoder.joblib"
)


print(
    "All artifacts loaded successfully."
)

All artifacts loaded successfully.


## 5. Validate threshold values

In [6]:
print(
    "Renewal threshold:",
    renewal_threshold
)

print(
    "Fraud threshold:",
    fraud_threshold
)


assert (
    0 <= float(
        renewal_threshold
    ) <= 1
), (
    "Renewal threshold must be between 0 and 1."
)


assert (
    0 <= float(
        fraud_threshold
    ) <= 1
), (
    "Fraud threshold must be between 0 and 1."
)


print(
    "Threshold validation passed."
)

Renewal threshold: 0.35
Fraud threshold: 0.65
Threshold validation passed.


## 6. Inspect saved feature schemas

In [7]:
def get_saved_features(
    model,
    model_name
):

    if hasattr(
        model,
        "feature_names_in_"
    ):

        features = list(
            model.feature_names_in_
        )

        print(
            f"{model_name}: "
            f"{len(features)} input features"
        )

        return features

    raise AttributeError(
        f"{model_name} does not expose feature_names_in_. "
        "Expected a fitted sklearn Pipeline."
    )


renewal_features = (
    get_saved_features(
        renewal_model,
        "Renewal Model"
    )
)

fraud_features = (
    get_saved_features(
        fraud_model,
        "Fraud Model"
    )
)

underwriting_features = (
    get_saved_features(
        underwriting_model,
        "Underwriting Model"
    )
)

Renewal Model: 54 input features
Fraud Model: 20 input features
Underwriting Model: 8 input features


In [8]:
print(
    "\nRenewal features:"
)
print(
    renewal_features
)

print(
    "\nFraud features:"
)
print(
    fraud_features
)

print(
    "\nUnderwriting features:"
)
print(
    underwriting_features
)


Renewal features:
['AGE', 'ANNUAL_INCOME', 'CREDIT_SCORE', 'SUM_INSURED', 'ANNUAL_PREMIUM', 'RISK_SCORE', 'DISCOUNT_PCT', 'COMMISSION_PCT', 'POLICY_TERM_DAYS', 'PREMIUM_TO_SUM_INSURED_PCT', 'HEALTH_SCORE', 'BMI', 'UNDERWRITING_SCORE', 'PREMIUM_LOADING_PCT', 'OLD_PREMIUM', 'NEW_PREMIUM', 'PREMIUM_INCREASE_PCT', 'PREMIUM_CHANGE_AMOUNT', 'NEW_PREMIUM_TO_INCOME_PCT', 'LOYALTY_YEARS', 'TOTAL_DUE_TXNS_PRIOR', 'PAID_TXNS_PRIOR', 'UNPAID_TXNS_AT_RENEWAL', 'TOTAL_PREMIUM_DUE_PRIOR', 'TOTAL_PREMIUM_PAID_PRIOR', 'AVG_PAYMENT_DELAY_PRIOR', 'MAX_PAYMENT_DELAY_PRIOR', 'PAYMENT_SUCCESS_RATE_PRIOR', 'TOTAL_CLAIMS_PRIOR', 'TOTAL_CLAIM_AMOUNT_PRIOR', 'AVG_CLAIM_AMOUNT_PRIOR', 'MAX_CLAIM_AMOUNT_PRIOR', 'DAYS_SINCE_LAST_CLAIM', 'HAS_PAYMENT_HISTORY_PRIOR', 'HAS_CLAIM_HISTORY_PRIOR', 'HAS_UNPAID_PAYMENT_AT_RENEWAL', 'HAS_UNDERWRITING_RECORD', 'PREMIUM_INCREASED_FLAG', 'HIGH_PREMIUM_INCREASE_FLAG', 'GENDER', 'MARITAL_STATUS', 'OCCUPATION', 'STATE', 'CUSTOMER_RISK_SEGMENT', 'POLICY_TYPE', 'SALES_CHANNEL', '

## 7. Validate underwriting classes

In [9]:
underwriting_classes = list(
    underwriting_label_encoder.classes_
)

print(
    "Underwriting classes:",
    underwriting_classes
)


expected_classes = {
    "Approved",
    "Approved with Loading",
    "Declined",
}


assert (
    set(
        underwriting_classes
    )
    ==
    expected_classes
), (
    "Unexpected underwriting target classes."
)


print(
    "Underwriting label validation passed."
)

Underwriting classes: ['Approved', 'Approved with Loading', 'Declined']
Underwriting label validation passed.


## 8. Load curated model datasets

In [10]:
renewal_df = pd.read_csv(
    CURATED_DIR
    / "renewal_model_dataset.csv",
    low_memory=False
)

fraud_df = pd.read_csv(
    CURATED_DIR
    / "fraud_model_dataset.csv",
    low_memory=False
)

underwriting_df = pd.read_csv(
    CURATED_DIR
    / "underwriting_model_dataset.csv",
    low_memory=False
)


print(
    "Renewal:",
    renewal_df.shape
)

print(
    "Fraud:",
    fraud_df.shape
)

print(
    "Underwriting:",
    underwriting_df.shape
)

Renewal: (19204, 60)
Fraud: (75000, 24)
Underwriting: (50000, 12)


## 9. Validate model features against saved datasets

In [11]:
def validate_features(
    df,
    features,
    model_name
):

    missing = [
        col
        for col in features
        if col not in df.columns
    ]

    print(
        model_name,
        "missing features:",
        missing
    )

    assert not missing, (
        f"{model_name} dataset is missing features: "
        f"{missing}"
    )


validate_features(
    renewal_df,
    renewal_features,
    "Renewal"
)

validate_features(
    fraud_df,
    fraud_features,
    "Fraud"
)

validate_features(
    underwriting_df,
    underwriting_features,
    "Underwriting"
)

Renewal missing features: []
Fraud missing features: []
Underwriting missing features: []


## 10. Renewal smoke prediction

In [12]:
renewal_sample = (
    renewal_df[
        renewal_features
    ]
    .iloc[
        [0]
    ]
)


renewal_probability = float(
    renewal_model
    .predict_proba(
        renewal_sample
    )[0, 1]
)


renewal_high_risk = int(
    renewal_probability
    >= float(
        renewal_threshold
    )
)


print(
    "Renewal non-renewal probability:",
    round(
        renewal_probability,
        4
    )
)

print(
    "High-risk flag:",
    renewal_high_risk
)

Renewal non-renewal probability: 0.4469
High-risk flag: 1


## 11. Fraud smoke prediction

In [13]:
fraud_sample = (
    fraud_df[
        fraud_features
    ]
    .iloc[
        [0]
    ]
)


fraud_probability = float(
    fraud_model
    .predict_proba(
        fraud_sample
    )[0, 1]
)


fraud_alert = int(
    fraud_probability
    >= float(
        fraud_threshold
    )
)


print(
    "Fraud probability:",
    round(
        fraud_probability,
        4
    )
)

print(
    "Fraud alert:",
    fraud_alert
)

print(
    "Reminder: alert means investigate, "
    "not proof of fraud."
)

Fraud probability: 0.1384
Fraud alert: 0
Reminder: alert means investigate, not proof of fraud.


## 12. Underwriting smoke prediction

In [14]:
underwriting_sample = (
    underwriting_df[
        underwriting_features
    ]
    .iloc[
        [0]
    ]
)


uw_probabilities = (
    underwriting_model
    .predict_proba(
        underwriting_sample
    )[0]
)


uw_class_id = int(
    np.argmax(
        uw_probabilities
    )
)


uw_decision = (
    underwriting_label_encoder
    .inverse_transform(
        [
            uw_class_id
        ]
    )[0]
)


uw_probability_dict = {
    class_name:
        round(
            float(probability),
            4
        )

    for class_name, probability
    in zip(
        underwriting_label_encoder.classes_,
        uw_probabilities
    )
}


print(
    "Underwriting decision:",
    uw_decision
)

print(
    "Probabilities:",
    uw_probability_dict
)

Underwriting decision: Approved with Loading
Probabilities: {'Approved': 0.369, 'Approved with Loading': 0.6227, 'Declined': 0.0083}


## 13. Validate predictor source files

In [15]:
source_checks = {
    "src/renewal_predictor.py":
        (
            PROJECT_DIR
            / "src"
            / "renewal_predictor.py"
        ),

    "src/models/fraud_predictor.py":
        (
            PROJECT_DIR
            / "src"
            / "models"
            / "fraud_predictor.py"
        ),

    "src/models/underwriting_predictor.py":
        (
            PROJECT_DIR
            / "src"
            / "models"
            / "underwriting_predictor.py"
        ),
}


source_status = []

for name, path in source_checks.items():

    source_status.append({
        "FILE":
            name,

        "EXISTS":
            path.exists(),
    })


source_status_df = (
    pd.DataFrame(
        source_status
    )
)

display(
    source_status_df
)

,FILE,EXISTS
0,src/renewal_predictor.py,True
1,src/models/fraud_predictor.py,True
2,src/models/underwriting_predictor.py,False


### Interpretation

At this point:

- missing `.joblib` files = ML artifact gap
- failed smoke prediction = training / preprocessing mismatch
- missing predictor source file = application-layer gap

The next stage fixes any application-layer gaps and then resumes:

```text
Predictors
→ Services
→ Decision Policy / HITL
→ FastAPI endpoints
→ Agentic AI Orchestrator
→ GUI
```

## 14. Final readiness report

In [16]:
readiness = pd.DataFrame(
    [
        {
            "MODULE":
                "Renewal",

            "MODEL_LOADED":
                True,

            "FEATURE_SCHEMA":
                True,

            "SMOKE_PREDICTION":
                True,
        },

        {
            "MODULE":
                "Fraud",

            "MODEL_LOADED":
                True,

            "FEATURE_SCHEMA":
                True,

            "SMOKE_PREDICTION":
                True,
        },

        {
            "MODULE":
                "Underwriting",

            "MODEL_LOADED":
                True,

            "FEATURE_SCHEMA":
                True,

            "SMOKE_PREDICTION":
                True,
        },
    ]
)


display(
    readiness
)


print()
print(
    "ALL THREE ML MODULES ARE READY "
    "FOR APPLICATION INTEGRATION."
)

,MODULE,MODEL_LOADED,FEATURE_SCHEMA,SMOKE_PREDICTION
0,Renewal,True,True,True
1,Fraud,True,True,True
2,Underwriting,True,True,True



ALL THREE ML MODULES ARE READY FOR APPLICATION INTEGRATION.


# Stage 15 Complete

If this notebook reaches the final message without errors, the missing ML gap is closed.

## Resume point

Continue from the application layer:

1. Ensure `src/models/underwriting_predictor.py`
2. Test `src/models/fraud_predictor.py`
3. Create/check:
   - `src/services/renewal_service.py`
   - `src/services/fraud_service.py`
   - `src/services/underwriting_service.py`
4. Complete `src/policies/decision_policy.py`
5. Complete `api/schemas.py`
6. Expose:
   - `GET /predict/renewal/{policy_id}`
   - `POST /predict/fraud`
   - `POST /predict/underwriting`
7. Add human review queue
8. Then continue to the Agentic Insurance AI Orchestrator and GUI.